# Практическое занятие: Композиция vs наследование, внедрение зависимоcтей

## Теория

**Проблема наследования:**  
- Жёсткая связь, хрупкость (изменение в базовом классе ломает потомков)  
- Нельзя изменить поведение в рантайме  
- Затруднено тестирование  

**Композиция:** объект содержит другие объекты (поведения) — "has‑a" вместо "is‑a".

**Dependency Injection (DI):** зависимости передаются извне (обычно через `__init__`), а не создаются внутри.

**Инверсия зависимостей (DIP):** код должен зависеть от абстракций, а не от конкретных классов. В Python абстракции — это `Protocol` (утиная типизация с проверкой).

**Паттерны в Python** благодаря композиции и DI реализуются проще (Стратегия, Декоратор, Команда — это по сути композиция поведения).

---

## Задача для разбора

### Задача 1. Система уведомлений (IS-A vs HAS-A)
Условие: Разработайте класс `Notifier`, который может отправлять сообщения через Email, SMS или Telegram. Обсудите недостатки наследования и перепишите с композицией + DI.




In [1]:
# Решение с наследованием (плохо)
class Notifier:
    def send(self, msg): raise NotImplementedError

class EmailNotifier(Notifier):
    def send(self, msg): print(f"Email: {msg}")

class SMSNotifier(Notifier):
    def send(self, msg): print(f"SMS: {msg}")

# Недостатки: дублирование кода, нельзя комбинировать, сложно добавлять новый канал

# ---------- Лучшее решение: композиция + DI ----------
from typing import Protocol

class MessageSender(Protocol):
    def send(self, msg: str) -> None: 
        ...

class EmailSender:
    def send(self, msg: str) -> None:
        print(f"📧 Email: {msg}")

class SMSSender:
    def send(self, msg: str) -> None:
        print(f"📱 SMS: {msg}")

class Notifier:
    def __init__(self, sender: MessageSender):
        self._sender = sender   # композиция, DI
        # self._sender = SMSSender()   # композиция, DI
    
    def notify(self, message: str):
        self._sender.send(message)

# Использование
notifier = Notifier(EmailSender())
notifier.notify("Привет!")

# Легко тестировать
class MockSender:
    def __init__(self):
        self.sent = []
    def send(self, msg): 
        self.sent.append(msg)

mock = MockSender()
test_notifier = Notifier(mock)
test_notifier.notify("Тест")
assert mock.sent == ["Тест"]
print("Тест пройден!")

📧 Email: Привет!
Тест пройден!


### Задача 2. Паттерн «Стратегия» через композицию
Условие: Реализовать калькулятор стоимости доставки, где стратегия расчёта может быть: `StandardDelivery`, `ExpressDelivery`, `Pickup`. Показать через композицию и протокол.



In [ ]:

from typing import Protocol

class DeliveryStrategy(Protocol):
    def calculate(self, order_total: float) -> float: 
        ...

class StandardDelivery:
    def calculate(self, order_total: float) -> float:
        return 5.0 if order_total < 50 else 0.0

class ExpressDelivery:
    def calculate(self, order_total: float) -> float:
        return 15.0

class Pickup:
    def calculate(self, order_total: float) -> float:
        return 0.0

class Order:
    def __init__(self, total: float, delivery: DeliveryStrategy):
        self.total = total
        self._delivery = delivery      # композиция, стратегия
    
    def total_cost(self) -> float:
        delivery_cost = self._delivery.calculate(self.total)
        return self.total + delivery_cost

# Пример
order = Order(30, StandardDelivery())
print(order.total_cost())   # 35.0

express_order = Order(100, ExpressDelivery())
print(express_order.total_cost())  # 115.0

# Динамическая смена стратегии
order._delivery = Pickup()
print(order.total_cost())   # 30.0

## 3 лёгкие задачи

**Лёгкая 1. Фильтр строк**  
Создайте класс `StringProcessor`, который в конструкторе получает объект с методом `process(text: str) -> str`. Реализуйте два конкретных процессора: `UpperCaseProcessor` и `StripProcessor`. Продемонстрируйте работу.

**Лёгкая 2. Логгер с разными выводами**  
Класс `Logger` должен уметь писать сообщения в разные места (файл, консоль). Сделайте через композицию: `ConsoleWriter`, `FileWriter`. Покажите, как переключать вывод в рантайме.

**Лёгкая 3. Автомобиль с двигателем**  
Класс `Car` получает в конструкторе двигатель (`Engine`). Реализуйте `PetrolEngine` и `ElectricEngine`. Метод `start()` делегирует вызов двигателю.

---



## 2 сложные задачи

### Сложная 1. Паттерн «Декоратор» через композицию

**Описание:**  
Реализовать систему наценок для кофе. Базовый кофе (эспрессо) — стоимость 100 руб. Декораторы (молоко +20, сироп +30, корица +10) могут динамически добавляться. Используйте композицию (каждый декоратор оборачивает другой объект). Не используйте наследование от `Coffee` (можно, но по условию нужно показать композицию: каждый декоратор получает в конструкторе `Coffee` и делегирует + добавляет стоимость).



In [ ]:
# Шаблон для сложной задачи 1

from typing import Protocol

class Coffee(Protocol):
    def cost(self) -> int: ...

class Espresso:
    def cost(self) -> int:
        return 100

# TODO: Реализуйте классы Milk, Syrup, Cinnamon как композиторы
# Каждый принимает coffee: Coffee в __init__ и в cost() возвращает coffee.cost() + свою цену

class Milk:
    def __init__(self, coffee: Coffee):
        self._coffee = coffee
    
    def cost(self) -> int:
        # вернуть стоимость кофе + 20
        ...

class Syrup:
    def __init__(self, coffee: Coffee):
        self._coffee = coffee
    
    def cost(self) -> int:
        # вернуть стоимость кофе + 30
        ...

class Cinnamon:
    def __init__(self, coffee: Coffee):
        self._coffee = coffee
    
    def cost(self) -> int:
        # вернуть стоимость кофе + 10
        ...

# Пример использования (должно работать после реализации):
# coffee = Milk(Syrup(Espresso()))
# print(coffee.cost())  # 150

### Сложная 2. Dependency Injection контейнер вручную

**Описание:**  
Создайте простой DI контейнер, который умеет регистрировать сервисы с их зависимостями. Реализуйте:
- `register(класс, зависимости)` — где зависимости — это список классов (или имён) других зарегистрированных сервисов.
- `resolve(класс)` — создаёт объект, рекурсивно создавая все его зависимости.

```python
# Шаблон для сложной задачи 2

class DIContainer:
    def __init__(self):
        self._services = {}   # class -> list of dependency classes
    
    def register(self, cls, dependencies: list):
        """регистрирует класс с его зависимостями (другими классами)"""
        self._services[cls] = dependencies
    
    def resolve(self, cls):
        # TODO: получить зависимости для cls, для каждой зависимости рекурсивно resolve
        # затем создать экземпляр cls, передав разрешённые объекты в конструктор
        pass

# Пример для проверки
class A: pass
class B:
    def __init__(self, a: A): self.a = a
class C:
    def __init__(self, b: B): self.b = b

container = DIContainer()
container.register(A, [])
container.register(B, [A])
container.register(C, [B])

c = container.resolve(C)
assert isinstance(c.b, B) and isinstance(c.b.a, A)
print("Готово")

## Домашнее задание

**Домашка 1. Фильмы и рейтинги**  
Создайте систему для расчёта рейтинга фильма. Рейтинг может вычисляться разными стратегиями: `KinoPoiskRating`, `IMDBRating`, `CriticsRating` (среднее от всех). Класс `Movie` получает стратегию и метод `get_rating()`. Реализуйте также возможность комбинирования (например, среднее арифметическое двух стратегий — паттерн Композиция). Продемонстрируйте гибкость.

**Домашка 2. Система оплаты (паттерн Команда + DI)**  
Класс `PaymentProcessor` принимает в конструкторе список объектов команд (`PayPalPayment`, `CardPayment`, `CryptoPayment`). Метод `pay(amount)` проходит по всем командам и пытается выполнить оплату, пока одна не успеет (или все). Реализуйте логику, что каждая команда имеет метод `try_pay(amount) -> bool`. Используйте композицию, DI, протокол для команды.

**Домашка 3. Переписать классическое наследование на композицию**  
Возьмите иерархию из реального проекта (или пример: `Animal -> Bird -> Sparrow + Ostrich`, `Vehicle -> Car -> ElectricCar + GasolineCar`). Перепишите так, чтобы использовалась композиция и внедрение зависимостей (например, поведение `FlyBehavior`, `EngineBehavior`). Добавьте простой DI контейнер для создания этих объектов. Объясните, как изменилась тестируемость и гибкость.